# DOWNLOADING DATA
For this project I only downloaded the flux, the flux error, and the bad pixel mask for inputs, and the effective temperature, log surface gravity, and metallicity for pontential outputs to classify the stars.

In [1]:
import pandas as pd
import numpy as np
import requests
import os
from astropy.io import fits
import io
import time

In [2]:
# Shuffling data for machine learning
df = pd.read_csv('cleaned_data.csv')
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)
df_shuffled.to_csv('cleaned_data_shuffled.csv', index=False)

In [3]:
# CONSTANTS
base_url = "https://data.sdss.org/sas/dr17/apogee/spectro/redux/dr17/stars"
csv_path = 'cleaned_data_shuffled.csv'
processed_data_path = 'processed_spectra.npy'
labels_path = 'processed_labels.npy'

df = pd.read_csv(csv_path)

# Resume logic
if os.path.exists(processed_data_path) and os.path.exists(labels_path):
    all_spectra = list(np.load(processed_data_path))
    all_labels = list(np.load(labels_path))
    start_idx = len(all_spectra) 
    print(f"Resuming from star index {start_idx} (already have {len(all_spectra)} stars)...")
else:
    all_spectra = []
    all_labels = []
    start_idx = 0

session = requests.Session()

for idx, row in df.iloc[start_idx:].iterrows():
    start_time = time.time()
    url = f"{base_url}/{row['TELESCOPE']}/{row['FIELD']}/{row['FILE']}"
    
    try:
        r = session.get(url, timeout=10) 
        
        if r.status_code == 200:
            with fits.open(io.BytesIO(r.content)) as hdul:
                flux = hdul[1].data[0]
                error = hdul[2].data[0]
                mask = hdul[3].data[0]
                
                # Handling bad pixels and gaps
                good = (mask == 0) & (error > 0) & (np.isfinite(flux))
                
                if np.sum(good) < 100:
                    print(f"[{idx}] SKIPPING: Too few good pixels for {row['FILE']}")
                    continue
                    
                med = np.nanmedian(flux[good])
                
                # Standardizing flux
                spec = np.stack([
                    np.where(good, flux/med, 1.0),
                    np.where(good, error/med, 10.0)
                ]).astype(np.float32)
                
                # Append data
                all_spectra.append(spec)
                all_labels.append([row['RV_TEFF'], row['RV_FEH'], row['RV_LOGG']])
            
                elapsed = time.time() - start_time
                print(f"[{idx}] Success ({len(all_spectra)} total) | {elapsed:.2f}s | {row['FILE']}")

        else:
            print(f"[{idx}] FAILED: Status {r.status_code} for {row['FILE']}")

        # Save every 100 successful stars
        if len(all_spectra) % 100 == 0 and r.status_code == 200:
            print(f"--- Checkpoint: Saving {len(all_spectra)} stars ---")
            np.save(processed_data_path, np.array(all_spectra))
            np.save(labels_path, np.array(all_labels))

    except Exception as ex:
        print(f"[{idx}] ERROR: {ex}")

# Final Save
print("Finalizing all files...")
final_data = np.array(all_spectra, dtype=np.float32)
final_labels = np.array(all_labels, dtype=np.float32)

np.save(processed_data_path, final_data)
np.save(labels_path, final_labels)

print(f"Done! Final shapes: Spectra {final_data.shape}, Labels {final_labels.shape}")

Resuming from star index 400 (already have 400 stars)...
[400] Success (401 total) | 0.82s | apStar-dr17-2M06550372+1803060.fits
[401] Success (402 total) | 0.39s | apStar-dr17-2M20451277-0113047.fits
[402] Success (403 total) | 0.30s | apStar-dr17-2M06242130-0454368.fits
[403] Success (404 total) | 0.32s | apStar-dr17-2M05381322-0021120.fits
[404] Success (405 total) | 0.16s | asStar-dr17-2M05214524-6202462.fits
[405] Success (406 total) | 0.16s | asStar-dr17-2M06250137-5955102.fits
[406] Success (407 total) | 0.94s | apStar-dr17-2M01542042+8459352.fits
[407] Success (408 total) | 0.39s | apStar-dr17-2M10301916+1654029.fits
[408] Success (409 total) | 0.47s | apStar-dr17-2M18530099+4630434.fits
[409] Success (410 total) | 0.61s | apStar-dr17-2M07170621+2754486.fits
[410] Success (411 total) | 0.81s | apStar-dr17-2M03155731+4835189.fits
[411] Success (412 total) | 1.14s | apStar-dr17-2M03172517+4838440.fits
[412] Success (413 total) | 1.03s | apStar-dr17-2M08033201+3900315.fits
[413] S

KeyboardInterrupt: 